In [1]:
%pip install xbbg

Looking in indexes: https://repository.saas.cagip.group.gca/artifactory/api/pypi/pypi-remote/simple
Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: xbbg in c:\program64\python\python\lib\site-packages (0.8.2)



In [2]:
%pip install --index-url=https://blpapi.bloomberg.com/repository/releases/python/simple/ blpapi

Looking in indexes: https://blpapi.bloomberg.com/repository/releases/python/simple/
Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install --upgrade pip

Looking in indexes: https://repository.saas.cagip.group.gca/artifactory/api/pypi/pypi-remote/simple
Note: you may need to restart the kernel to use updated packages.


In [6]:
from xbbg import blp 
import pandas as pd 
import numpy as np 
from datetime import datetime 
import blpapi

In [8]:
def get_swaption_data():
    # Initialize session
    sessionOptions = blpapi.SessionOptions()
    sessionOptions.setServerHost("localhost")
    sessionOptions.setServerPort(8194)
    session = blpapi.Session(sessionOptions)
    
    if not session.start():
        print("Failed to start session.")
        return
    
    if not session.openService("//blp/refdata"):
        print("Failed to open //blp/refdata")
        return
    
    refDataService = session.getService("//blp/refdata")
    request = refDataService.createRequest("ReferenceDataRequest")
    
    swaptions = [
        'USSW2 2.008 Straddle Curncy', # 2Y Swaption at 2.008%
        'USSW30 ATM Call Curncy'
    ]
    
    for swaption in swaptions:
        request.append('securities', swaptions)
        
    # fields to retrieve 
    fields = [
        'OPT_STRIKE_PX', # Strike Price
        'OPT_EXPIRE_DT', # Expiry Date
        'OPT_PUT_CALL', # Put/Call indicator
        'OPT_DELTA', # Delta
        'OPT_UNDL_TENOR', # Underlying Swap Tenor
        'PX_LAST', # Last Price
        'OPT_NOTIONAL_AMT', # Notional Amount
        'OPT_VOLATILIY', # Implied Vol    
    ]
    for field in fields:
        request.append("fields", field)
    
    session.sendRequest(request)
    
    # Process response
    while True:
        ev = session.nextEvent(500)
        for msg in ev:
            if msg.hasElement("securityData"):
                securityDataArray = msg.getElement("securityData")
                for i in range(securityDataArray.numValues()):
                    securityData = securityDataArray.getValueAsElement(i)
                    print(f"\nSecurity: {securityData.getElementAsString('security')}")
                    
                    fieldData = securityData.getElement("fieldData")
                    for field in fields:
                        if fieldData.hasElement(field):
                            print(f"{field}: {fieldData.getElement(field).getValue()}")
        
        if ev.eventType() == blpapi.Event.RESPONSE:
            break
    
    session.stop()

if __name__ == "__main__":
    get_swaption_data()



Security: ['USSW2 2.008 Straddle Curncy', 'USSW30 ATM Call Curncy']

Security: ['USSW2 2.008 Straddle Curncy', 'USSW30 ATM Call Curncy']


# Swaption Delta Calculator 

In [16]:
from xbbg import blp
import numpy as np
from scipy.stats import norm
import pandas as pd
from datetime import datetime


In [ ]:
import numpy as np
from scipy.stats import norm
from xbbg import blp

class SwaptionDeltaCalculator:
    """
    Real-time Swaption Delta Calculator using Bloomberg
    """
    
    def __init__(self):
        self.swap_rates = {}
        self.swaption_vols = {}
        
    def get_market_data(self, swap_tenors=['2', '5', '10', '30'], 
                        vol_expiries=['1M', '3M', '6M', '1', '2']):
        """
        Get the real time market data for the swap tenors and the swaption expiries
        """
        
        print('Fetching real time market data')
        print('=' * 80)
        
        # Get swap rates
        swap_tickers = [f'USSP{tenor} Curncy' for tenor in swap_tenors]
        
        try:
            swap_data = blp.bdp(swap_tickers, ['PX_LAST', 'PX_BID', 'PX_ASK'])
            print('\nSwap Rates:')
            print(swap_data)
            
            for i, tenor in enumerate(swap_tenors):
                ticker = swap_tickers[i]
                if ticker in swap_data.index:
                    self.swap_rates[tenor] = {
                        'mid': swap_data.loc[ticker, 'PX_LAST'],
                        'bid': swap_data.loc[ticker, 'PX_BID'],
                        'ask': swap_data.loc[ticker, 'PX_ASK'],
                    }
        except Exception as e:
            print(f"Error fetching swap rates: {e}")
            
        # Get swaption Vols
        vol_tickers = []
        vol_labels = []
        
        for expiry in vol_expiries:
            for tenor in swap_tenors:
                # FIX 1: Remove spaces in ticker format
                ticker = f"USSV{expiry}x{tenor}Y Curncy"
                vol_tickers.append(ticker)
                vol_labels.append(f"{expiry}x{tenor}Y")
                
        try:
            vol_data = blp.bdp(vol_tickers, ['PX_LAST', 'PX_BID', 'PX_ASK'])
            print('\nSwaption Implied Vols:')
            print(vol_data)
            
            # Store vols
            for i, label in enumerate(vol_labels):
                ticker = vol_tickers[i]
                if ticker in vol_data.index:
                    self.swaption_vols[label] = {
                        'mid': vol_data.loc[ticker, 'PX_LAST'],
                        'bid': vol_data.loc[ticker, 'PX_BID'],
                        'ask': vol_data.loc[ticker, 'PX_ASK']
                    }
                    
        except Exception as e:
            print(f'Error fetching swaption vols: {e}')
            
        print('\n' + '=' * 80)
        
        return self.swap_rates, self.swaption_vols
            
    def calculate_annuity(self, swap_rate, tenor_years, frequency=2):
        """
        Calculating the Swap annuity factor
        
        Parameters:
        -----------
        swap_rate: float (percentage)
        tenor_years: int (swap tenor in years)
        frequency: int (payment frequency per year, semi-annual = 2)      
        """
        
        rate = swap_rate / 100
        n_periods = tenor_years * frequency
        period_rate = rate / frequency
        
        annuity = sum([1/(1 + period_rate)**(i+1) for i in range(n_periods)])
        
        # FIX 2: Add return statement and divide by frequency
        return annuity / frequency
        
    def calculate_delta(self,
                        forward_rate,
                        strike,
                        volatility,
                        time_to_expiry_years,
                        swap_tenor_years,
                        option_type='payer',
                        notional=1_000_000,
                        output_type='dv01'):
        """
        Calculate swaption delta using Black's model
        
        Parameters:
        -----------
        forward_rate : float
            Forward swap rate in % (e.g., 3.5)
        strike : float
            Strike rate in % (e.g., 3.0)
        volatility : float
            Implied volatility in % (e.g., 50 for 50%)
        time_to_expiry_years : float
            Time to expiry in years
        swap_tenor_years : int
            Underlying swap tenor in years
        option_type : str
            'payer' (call on rates) or 'receiver' (put on rates)
        notional : float
            Notional amount
        output_type : str
            'dv01' (dollar value per 1bp) or 'percentage' (0-1)
        
        Returns:
        --------
        dict with delta calculations
        """
        
        # Convert to Decimals
        F = forward_rate / 100
        K = strike / 100
        T = time_to_expiry_years
        sigma = volatility / 100
        
        # FIX 3: Change T<0 to T<=0
        if T <= 0:
            if option_type.lower() == 'payer':
                delta_pct = 1.0 if F > K else 0.0
            else:
                delta_pct = 1.0 if F < K else 0.0
                
            annuity = self.calculate_annuity(forward_rate, swap_tenor_years)
            dv01 = delta_pct * annuity * notional * 0.0001
            
            return {
                'delta_percentage': delta_pct,
                'dv01': dv01,
                'annuity': annuity,
                'status': 'expired'
            } 
            
        # Black's Formula
        d1 = (np.log(F/K) + 0.5 * sigma**2 * T) / (sigma * np.sqrt(T))
        d2 = d1 - sigma * np.sqrt(T)
        
        # Calculate Delta 
        if option_type.lower() == 'payer':
            delta_pct = norm.cdf(d1)
        elif option_type.lower() == 'receiver':
            delta_pct = norm.cdf(-d1)
        else:
            raise ValueError('option_type must be payer or receiver')
        
        # Calculate Annuity
        annuity = self.calculate_annuity(forward_rate, swap_tenor_years)
        
        # FIX 4: Fix typo dvo1 -> dv01
        dv01 = delta_pct * annuity * notional * 0.0001
        
        # Additional Greeks
        vega = F * annuity * notional * norm.pdf(d1) * np.sqrt(T) * 0.01  # per 1% vol
        gamma = norm.pdf(d1) / (F * sigma * np.sqrt(T))
        
        return {
            'delta_percentage': delta_pct,
            'dv01': dv01,
            'annuity': annuity,
            'vega': vega,
            'gamma': gamma,
            'd1': d1,
            'd2': d2,
            'status': 'active'
        }
        
    def calculate_straddle_delta(self, forward_rate, strike, volatility, 
                                 time_to_expiry_years, swap_tenor_years,
                                 notional=1_000_000):
        """
        Calculate delta for straddle (payer + receiver)
        """
        
        payer = self.calculate_delta(
            forward_rate, strike, volatility, time_to_expiry_years, 
            swap_tenor_years, 'payer', notional
        )
        
        receiver = self.calculate_delta(
            forward_rate, strike, volatility, time_to_expiry_years, 
            swap_tenor_years, 'receiver', notional
        )
        
        return {
            # FIX 5: Fix typo 'player' -> 'payer'
            'payer_delta_pct': payer['delta_percentage'],
            'receiver_delta_pct': receiver['delta_percentage'],
            'straddle_delta_pct': payer['delta_percentage'] - receiver['delta_percentage'],
            'payer_dv01': payer['dv01'],
            # FIX 6: Fix key name 'receiver dv01' -> 'receiver_dv01'
            'receiver_dv01': receiver['dv01'],
            'straddle_dv01': payer['dv01'] + receiver['dv01'],
            'payer': payer,
            'receiver': receiver
        }    
        
    def price_swaption(self, forward_rate, strike, volatility, 
                      time_to_expiry_years, swap_tenor_years, 
                      option_type='payer', notional=1_000_000):
        """
        Price swaption using Black's model
        """
        
        F = forward_rate / 100
        K = strike / 100
        sigma = volatility / 100
        T = time_to_expiry_years    
        
        # Handle Expiry
        if T <= 0:
            if option_type.lower() == 'payer':
                intrinsic = max(F - K, 0)
            else: 
                intrinsic = max(K - F, 0)
            
            annuity = self.calculate_annuity(forward_rate, swap_tenor_years)
            price = intrinsic * annuity * notional
            
            return price
            
        # Black's formula
        d1 = (np.log(F/K) + 0.5 * sigma**2 * T) / (sigma * np.sqrt(T))
        d2 = d1 - sigma * np.sqrt(T)
        
        annuity = self.calculate_annuity(forward_rate, swap_tenor_years)
        
        if option_type.lower() == 'payer':
            price = annuity * notional * (F * norm.cdf(d1) - K * norm.cdf(d2))
        # FIX 7: Add receiver case
        else:
            price = annuity * notional * (K * norm.cdf(-d2) - F * norm.cdf(-d1))
            
        return price 


# Example usage
def main():
    calc = SwaptionDeltaCalculator()
    
    # Get market data
    swap_rates, vols = calc.get_market_data(
        swap_tenors=['2', '5', '10', '30'],
        vol_expiries=['1M', '3M', '6M', '1', '2']
    )
    
    print("\n" + "=" * 80)
    print("EXAMPLE CALCULATIONS")
    print("=" * 80)
    
    # Example: Calculate delta for a 2Y swaption
    if '2' in swap_rates and '1x2Y' in vols:
        forward_2y = swap_rates['2']['mid']
        vol_2y = vols['1x2Y']['mid']
        
        print(f"\nExample: 1Y option on 2Y swap (Payer)")
        print(f"Forward Rate: {forward_2y:.3f}%")
        print(f"Volatility: {vol_2y:.2f}%")
        
        delta = calc.calculate_delta(
            forward_rate=forward_2y,
            strike=forward_2y,  # ATM
            volatility=vol_2y,
            time_to_expiry_years=1.0,
            swap_tenor_years=2,
            option_type='payer',
            notional=10_000_000
        )
        
        print(f"\nResults:")
        print(f"  Delta: {delta['delta_percentage']:.4f}")
        print(f"  DV01: ${delta['dv01']:,.2f}")
        print(f"  Vega: ${delta['vega']:,.2f}")

if __name__ == "__main__":
    main()

Fetching real time market data

Swap Rates:
Empty DataFrame
Columns: []
Index: []

Swaption Implied Vols:
Empty DataFrame
Columns: []
Index: []


EXAMPLE CALCULATIONS


In [34]:
# Initialize calculator
calc = SwaptionDeltaCalculator()

# Get market data
calc.get_market_data()

# Calculate your positions
position1 = calc.calculate_position_delta({
    'tenor': '2',
    'strike': 2.008,
    'option_type': 'straddle',
    'position': 'sell',
    'notional': 10_000_000,
    'time_to_expiry': 2/365,
    'volatility': 50.0
})

print(f"Position 1 DV01: ${position1['position_delta_dv01']:,.2f}")

Fetching real-time market data from Bloomberg...

Swap Rates (USSP format):
               px_last   px_bid   px_ask
USSP10 Curncy   1.7560   1.7560   1.7560
USSP2 Curncy   33.1509  33.1509  33.1509
USSP30 Curncy -43.5750 -43.5750 -43.5750
USSP5 Curncy    1.5158   1.5158   1.5158
  ✓ 2Y: 33.151%
  ✓ 5Y: 1.516%
  ✓ 10Y: 1.756%
  ✓ 30Y: -43.575%

Swaption Implied Volatilities:
Empty DataFrame
Columns: []
Index: []
  ⚠ No swaption volatility data available

Retrieved 4 swap rates and 0 swaption vols
Position 1 DV01: $-1,383.17


In [36]:
from xbbg import blp
import pandas as pd

def find_correct_swap_tickers():
    """
    Systematically find the correct US swap rate tickers
    """
    
    print("=" * 80)
    print("FINDING CORRECT US SWAP RATE TICKERS")
    print("=" * 80)
    
    # Test all common formats for 2Y swap
    test_formats = [
        # Format: (ticker, description)
        ('YCSW0002 Index', 'Yield Curve Swap 2Y'),
        ('USSWIT2 Curncy', 'US Swap 2Y'),
        ('USSW2 Curncy', 'US Swap 2Y Standard'),
        ('USSWAP2 Curncy', 'US Swap 2Y Alt'),
        ('USDR2T Curncy', 'USD 2Y Rate'),
        ('USISDA02 Index', 'ISDA 2Y'),
        ('S0023FS Index', 'SOFR Swap 2Y'),
        ('USOSFR2 Curncy', 'SOFR 2Y'),
        ('USSOFR2 Curncy', 'SOFR 2Y Alt'),
        ('USSP2 BGN Curncy', 'BGN 2Y'),
        ('USSP2 ICAP Curncy', 'ICAP 2Y'),
    ]
    
    working_tickers = []
    
    for ticker, description in test_formats:
        print(f"\nTesting: {ticker} ({description})")
        try:
            # Get name and price
            data = blp.bdp(ticker, ['NAME', 'PX_LAST'])
            
            if not data.empty:
                # Check if we have data
                if 'name' in data.columns:
                    name = data.loc[ticker, 'name']
                    px = data.loc[ticker, 'px_last'] if 'px_last' in data.columns else None
                    
                    if pd.notna(name):
                        print(f"  ✓ Found: {name}")
                        if pd.notna(px):
                            print(f"  ✓ Price: {px:.3f}%")
                            
                            # Sanity check: swap rates should be between 0-10%
                            if 0 < px < 10:
                                print(f"  ✓✓ LOOKS VALID!")
                                working_tickers.append((ticker, description, px))
                            else:
                                print(f"  ⚠ Price seems wrong (should be 0-10%)")
                        else:
                            print(f"  ⚠ No price data")
                else:
                    print(f"  ⚠ Ticker exists but no name field")
            else:
                print(f"  ✗ No data returned")
                
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print("\n" + "=" * 80)
    print("RESULTS:")
    print("=" * 80)
    
    if working_tickers:
        print(f"\n✓✓ Found {len(working_tickers)} valid swap rate ticker(s):\n")
        for ticker, desc, price in working_tickers:
            print(f"  {ticker:25s} {desc:20s} {price:.3f}%")
        
        # Test all tenors with the first working format
        print("\n" + "=" * 80)
        print("TESTING ALL TENORS WITH FIRST WORKING FORMAT:")
        print("=" * 80)
        
        base_ticker = working_tickers[0][0]
        # Extract the format pattern
        if '2' in base_ticker:
            test_all_tenors(base_ticker)
        
        return working_tickers
    else:
        print("\n✗ NO VALID SWAP RATE TICKERS FOUND")
        print("\nThis means:")
        print("1. You may not have swap rate data subscription")
        print("2. Need to use SWPM or IRSB to access rates")
        print("3. Need to check with Bloomberg support")
        return None


def test_all_tenors(base_ticker):
    """
    Test all tenors once we find the right format
    """
    
    # Replace '2' with placeholder
    if '02' in base_ticker:
        # Format like YCSW0002
        format_template = base_ticker.replace('02', '{:02d}')
        tenors = [1, 2, 3, 5, 7, 10, 15, 20, 30]
    elif '2' in base_ticker:
        # Format like USSW2
        format_template = base_ticker.replace('2', '{}')
        tenors = [1, 2, 3, 5, 7, 10, 15, 20, 30]
    else:
        print("Can't determine format pattern")
        return
    
    print(f"\nFormat template: {format_template}")
    print("\nTesting all tenors:")
    
    valid_tenors = {}
    
    for tenor in tenors:
        ticker = format_template.format(tenor)
        try:
            data = blp.bdp(ticker, 'PX_LAST')
            if not data.empty and 'px_last' in data.columns:
                px = data.loc[ticker, 'px_last']
                if pd.notna(px) and 0 < px < 10:
                    print(f"  ✓ {tenor:2d}Y: {px:.3f}% ({ticker})")
                    valid_tenors[tenor] = {'ticker': ticker, 'rate': px}
        except:
            pass
    
    return valid_tenors


def check_sofr_vs_libor():
    """
    Check if you have SOFR or LIBOR swap rates
    """
    
    print("\n" + "=" * 80)
    print("CHECKING SOFR VS LIBOR SWAP RATES")
    print("=" * 80)
    
    print("\nNote: US swaps transitioned from LIBOR to SOFR in 2023")
    print("You likely need SOFR swap rates, not LIBOR\n")
    
    # Test SOFR swap tickers
    sofr_tickers = [
        'S0023FS Index',      # SOFR 2Y
        'USOSFR2 Curncy',     # SOFR 2Y alt
        'USSOFR2 Curncy',     # SOFR 2Y alt2
    ]
    
    print("Testing SOFR swap tickers:")
    for ticker in sofr_tickers:
        try:
            data = blp.bdp(ticker, ['NAME', 'PX_LAST'])
            if not data.empty:
                print(f"\n  {ticker}:")
                print(f"    {data}")
        except:
            print(f"  ✗ {ticker}: Not found")


def manual_instructions():
    """
    Provide manual instructions for Bloomberg Terminal
    """
    
    print("\n" + "=" * 80)
    print("MANUAL BLOOMBERG TERMINAL INSTRUCTIONS")
    print("=" * 80)
    
    print("""
Since we can't find the correct tickers programmatically, please do this:

METHOD 1: Use IRSB (Interest Rate Swap Builder)
------------------------------------------------
1. In Bloomberg Terminal, type: IRSB<GO>
2. Select:
   - Currency: USD
   - Floating Index: SOFR (or whatever is default)
   - Tenor: 2Y
3. Click "Calculate" or "Price"
4. Look at the swap rate displayed
5. RIGHT-CLICK on the rate value
6. Select "Ticker" or "Copy Ticker"
7. **PASTE THAT TICKER HERE**

METHOD 2: Use SWPM (Swap Manager)
----------------------------------
1. Type: SWPM<GO>
2. Select "Swap" (not Swaption)
3. Build a 2Y USD swap
4. Look at the par rate
5. Right-click on the rate
6. Select "Ticker"
7. **PASTE THAT TICKER HERE**

METHOD 3: Search for Swap Curve
--------------------------------
1. Type: ALLX<GO>
2. Search: "USD SOFR swap curve"
3. Look at the results
4. Find the 2Y point
5. Note the ticker format
6. **PASTE THAT TICKER HERE**

METHOD 4: Check Your Swap Curve
--------------------------------
1. Type: ICVS<GO> (Intraday Curve Viewer)
2. Select USD swap curve
3. Look at the 2Y point
4. Right-click and get ticker
5. **PASTE THAT TICKER HERE**

Once you have the correct ticker, we can update the code!
""")


if __name__ == "__main__":
    # Run comprehensive search
    working = find_correct_swap_tickers()
    
    # Check SOFR vs LIBOR
    check_sofr_vs_libor()
    
    # Provide manual instructions
    manual_instructions()
    
    if working:
        print("\n" + "=" * 80)
        print("✓ SUCCESS! Use this ticker format in your code:")
        print("=" * 80)
        ticker = working[0][0]
        print(f"\nReplace 'USSP{{tenor}} Curncy' with:")
        if 'YCSW' in ticker:
            print(f"  'YCSW{{tenor:04d}} Index'")
        elif 'SOFR' in ticker:
            print(f"  'USOSFR{{tenor}} Curncy'")
        else:
            print(f"  '{ticker}' (replace 2 with {{tenor}})")
    else:
        print("\n" + "=" * 80)
        print("⚠ NEXT STEPS:")
        print("=" * 80)
        print("1. Follow the manual instructions above")
        print("2. Find the correct ticker in Bloomberg Terminal")
        print("3. Report back the ticker format")
        print("4. I'll update the code with the correct format")

FINDING CORRECT US SWAP RATE TICKERS

Testing: YCSW0002 Index (Yield Curve Swap 2Y)
  ✗ No data returned

Testing: USSWIT2 Curncy (US Swap 2Y)
  ✓ Found: USD INFL SWAP ZC 2Y
  ✓ Price: 2.458%
  ✓✓ LOOKS VALID!

Testing: USSW2 Curncy (US Swap 2Y Standard)
  ✓ Found: USD SWAP SEMI 30/360 2Y
  ⚠ No price data

Testing: USSWAP2 Curncy (US Swap 2Y Alt)
  ✓ Found: USD SWAP SA 30/360 2YR*
  ⚠ No price data

Testing: USDR2T Curncy (USD 2Y Rate)
  ✓ Found: USD DEPOSIT T/N
  ✓ Price: 3.675%
  ✓✓ LOOKS VALID!

Testing: USISDA02 Index (ISDA 2Y)
  ✓ Found: USD ICE SWAP RATE 11:00am NY 2
  ✓ Price: 5.087%
  ✓✓ LOOKS VALID!

Testing: S0023FS Index (SOFR Swap 2Y)
  ✓ Found: USD S23 Fwd Swap
  ⚠ No price data

Testing: USOSFR2 Curncy (SOFR 2Y)
  ✓ Found: USD OIS  ANN VS SOFR 2Y
  ✓ Price: 4.036%
  ✓✓ LOOKS VALID!

Testing: USSOFR2 Curncy (SOFR 2Y Alt)
  ✗ No data returned

Testing: USSP2 BGN Curncy (BGN 2Y)
  ✓ Found: USD SWAP SPREAD SEMI 2YR
  ⚠ No price data

Testing: USSP2 ICAP Curncy (ICAP 2Y)
  ✗ 

# Corrected with USOFR

In [44]:
import numpy as np
from scipy.stats import norm
from xbbg import blp
import pandas as pd

class SwaptionDeltaCalculator:
    """
    Clean Swaption Delta Calculator using Bloomberg data
    """
    
    def __init__(self):
        self.swap_rates = {}
        self.swaption_vols = {}
        
    def get_swap_rates(self, tenors=['2', '5', '10', '30']):
        """
        Fetch SOFR swap rates from Bloomberg
        
        Parameters:
        -----------
        tenors : list
            List of swap tenors (e.g., ['2', '5', '10', '30'])
        """
        tickers = [f'USOSFR{tenor} Curncy' for tenor in tenors]
        
        try:
            data = blp.bdp(tickers, 'PX_LAST')
            
            for i, tenor in enumerate(tenors):
                ticker = tickers[i]
                if ticker in data.index:
                    rate = data.loc[ticker, 'px_last' if 'px_last' in data.columns else 'PX_LAST']
                    if pd.notna(rate):
                        self.swap_rates[tenor] = rate
                        
            return self.swap_rates
            
        except Exception as e:
            print(f"Error fetching swap rates: {e}")
            return {}
    
    def set_volatilities(self, vol_dict):
        """
        Set swaption volatilities manually
        
        Parameters:
        -----------
        vol_dict : dict
            e.g., {'1Mx2': 0.7990, '1Mx30': 0.5309}
            Volatilities in % (79.90 bps = 0.7990)
        """
        self.swaption_vols = vol_dict
        
    def calculate_annuity(self, swap_rate, tenor_years, frequency=2):
        """
        Calculate swap annuity (PVBP)
        
        Parameters:
        -----------
        swap_rate : float
            Swap rate in % (e.g., 4.046)
        tenor_years : int
            Swap tenor in years
        frequency : int
            Payment frequency per year (2 = semi-annual)
        """
        rate = swap_rate / 100
        n_periods = tenor_years * frequency
        period_rate = rate / frequency
        
        annuity = sum([1/(1 + period_rate)**(i+1) for i in range(n_periods)])
        return annuity / frequency
        
    def calculate_delta(self, forward_rate, strike, volatility, 
                       time_to_expiry_years, swap_tenor_years,
                       option_type='payer', notional=1_000_000):
        """
        Calculate swaption delta using Black's model
        
        Parameters:
        -----------
        forward_rate : float
            Forward swap rate in % (e.g., 4.046)
        strike : float
            Strike rate in % (e.g., 4.008)
        volatility : float
            Implied volatility in % (e.g., 0.7990 for 79.90 bps)
        time_to_expiry_years : float
            Time to expiry in years (e.g., 2/365 for 2 days)
        swap_tenor_years : int
            Underlying swap tenor in years
        option_type : str
            'payer' or 'receiver'
        notional : float
            Notional amount in dollars
            
        Returns:
        --------
        dict with:
            - dv01: Delta in $ per 1bp move
            - delta_pct: Delta as percentage (0-1)
            - annuity: Swap annuity factor
            - d1, d2: Black's model parameters
        """
        
        # Convert to decimals
        F = forward_rate / 100
        K = strike / 100
        T = time_to_expiry_years
        sigma = volatility / 100
        
        # Handle expiry
        if T <= 0:
            if option_type.lower() == 'payer':
                delta_pct = 1.0 if F > K else 0.0
            else:
                delta_pct = -1.0 if F < K else 0.0
                
            annuity = self.calculate_annuity(forward_rate, swap_tenor_years)
            dv01 = delta_pct * annuity * notional * 0.0001
            
            return {
                'dv01': dv01,
                'delta_pct': delta_pct,
                'annuity': annuity,
                'd1': None,
                'd2': None
            }
        
        # Black's formula
        d1 = (np.log(F/K) + 0.5 * sigma**2 * T) / (sigma * np.sqrt(T))
        d2 = d1 - sigma * np.sqrt(T)
        
        # Calculate delta
        if option_type.lower() == 'payer':
            delta_pct = norm.cdf(d1)
        elif option_type.lower() == 'receiver':
            delta_pct = norm.cdf(d1) - 1.0
        else:
            raise ValueError("option_type must be 'payer' or 'receiver'")
        
        # Calculate annuity
        annuity = self.calculate_annuity(forward_rate, swap_tenor_years)
        
        # DV01 = delta × annuity × notional × 0.0001
        dv01 = delta_pct * annuity * notional * 0.0001
        
        return {
            'dv01': dv01,
            'delta_pct': delta_pct,
            'annuity': annuity,
            'd1': d1,
            'd2': d2
        }
    
    def calculate_straddle(self, forward_rate, strike, volatility,
                          time_to_expiry_years, swap_tenor_years,
                          notional=1_000_000):
        """
        Calculate straddle delta (long payer + long receiver)
        
        Returns:
        --------
        dict with:
            - dv01: Total straddle DV01
            - payer_dv01: Payer leg DV01
            - receiver_dv01: Receiver leg DV01
        """
        
        payer = self.calculate_delta(
            forward_rate, strike, volatility, time_to_expiry_years,
            swap_tenor_years, 'payer', notional
        )
        
        receiver = self.calculate_delta(
            forward_rate, strike, volatility, time_to_expiry_years,
            swap_tenor_years, 'receiver', notional
        )
        
        return {
            'dv01': payer['dv01'] + receiver['dv01'],
            'payer_dv01': payer['dv01'],
            'receiver_dv01': receiver['dv01'],
            'payer_delta_pct': payer['delta_pct'],
            'receiver_delta_pct': receiver['delta_pct'],
            'annuity': payer['annuity']
        }


# Example usage
def main():
    # Initialize calculator
    calc = SwaptionDeltaCalculator()
    
    # Get swap rates from Bloomberg
    print("Fetching swap rates...")
    swap_rates = calc.get_swap_rates(['2', '30'])
    print(f"2Y SOFR: {swap_rates.get('2', 'N/A'):.3f}%")
    print(f"30Y SOFR: {swap_rates.get('30', 'N/A'):.3f}%")
    
    # Set volatilities from VCUB
    calc.set_volatilities({
        '1Mx2': 0.7990,   # 79.90 bps
        '1Mx30': 0.5309   # 53.09 bps
    })
    
    print("\n" + "="*60)
    print("EXAMPLE 1: 2Y Straddle")
    print("="*60)
    
    # Calculate 2Y straddle delta
    result1 = calc.calculate_straddle(
        forward_rate=swap_rates['2'],
        strike=4.008,
        volatility=0.7990,
        time_to_expiry_years=2/365,
        swap_tenor_years=2,
        notional=150_000_000
    )
    
    print(f"Payer DV01:     ${result1['payer_dv01']:,.2f}")
    print(f"Receiver DV01:  ${result1['receiver_dv01']:,.2f}")
    print(f"Straddle DV01:  ${result1['dv01']:,.2f}")
    print(f"SOLD DV01:      ${-result1['dv01']:,.2f}")
    
    print("\n" + "="*60)
    print("EXAMPLE 2: 30Y Payer (ATM)")
    print("="*60)
    
    # Calculate 30Y payer delta
    result2 = calc.calculate_delta(
        forward_rate=swap_rates['30'],
        strike=swap_rates['30'],  # ATM
        volatility=0.5309,
        time_to_expiry_years=2/365,
        swap_tenor_years=30,
        option_type='payer',
        notional=25_000_000
    )
    
    print(f"Delta %:        {result2['delta_pct']:.4f}")
    print(f"DV01:           ${result2['dv01']:,.2f}")
    print(f"Annuity:        {result2['annuity']:.6f}")


if __name__ == "__main__":
    main()

Fetching swap rates...
2Y SOFR: 4.045%
30Y SOFR: 4.187%

EXAMPLE 1: 2Y Straddle
Payer DV01:     $28,542.38
Receiver DV01:  $0.00
Straddle DV01:  $28,542.38
SOLD DV01:      $-28,542.38

EXAMPLE 2: 30Y Payer (ATM)
Delta %:        0.5001
DV01:           $21,244.66
Annuity:        16.993063


In [46]:
def print_position_summary(positions):
    """
    Print a formatted summary table of positions
    
    Parameters:
    -----------
    positions : list of dict
        Each dict should contain position details and calculated results
    """
    
    print("\n" + "=" * 120)
    print("POSITION SUMMARY")
    print("=" * 120)
    
    # Header
    header = f"{'Position':<20} {'Type':<8} {'Strike':<10} {'Forward':<10} {'Moneyness':<12} {'Delta %':<10} {'DV01':<15} {'Interpretation':<25}"
    print(header)
    print("-" * 120)
    
    # Rows
    for pos in positions:
        row = (f"{pos['name']:<20} "
               f"{pos['type']:<8} "
               f"{pos['strike']:>8.3f}% "
               f"{pos['forward']:>8.3f}% "
               f"{pos['moneyness']:>10.0f}bp "
               f"{pos['delta_pct']:>8.1f}% "
               f"${pos['dv01']:>13,.0f} "
               f"{pos['interpretation']:<25}")
        print(row)
    
    print("-" * 120)
    
    # Net portfolio delta
    net_dv01 = sum([pos['dv01'] for pos in positions])
    print(f"\n{'Net Portfolio Delta:':<60} ${net_dv01:>13,.0f}")
    
    if net_dv01 > 0:
        print(f"{'Interpretation:':<60} Long rates (gain if rates rise)")
    elif net_dv01 < 0:
        print(f"{'Interpretation:':<60} Short rates (lose if rates rise)")
    else:
        print(f"{'Interpretation:':<60} Delta neutral")
    
    print("=" * 120)


# Example usage with your positions:
def main():
    calc = SwaptionDeltaCalculator()
    
    # Get swap rates
    swap_rates = calc.get_swap_rates(['2', '30'])
    calc.set_volatilities({'1Mx2': 0.7990, '1Mx30': 0.5309})
    
    # Calculate Position 1: 2Y Straddle (SOLD)
    result1 = calc.calculate_straddle(
        forward_rate=swap_rates['2'],
        strike=2.008,
        volatility=0.7990,
        time_to_expiry_years=2/365,
        swap_tenor_years=2,
        notional=150_000_000
    )
    
    # Calculate Position 2: 30Y Payer (BUY)
    result2 = calc.calculate_delta(
        forward_rate=swap_rates['30'],
        strike=swap_rates['30'],
        volatility=0.5309,
        time_to_expiry_years=2/365,
        swap_tenor_years=30,
        option_type='payer',
        notional=25_000_000
    )
    
    # Prepare data for table
    positions = [
        {
            'name': '1 (2Y Straddle)',
            'type': 'SELL',
            'strike': 2.008,
            'forward': swap_rates['2'],
            'moneyness': (2.008 - swap_rates['2']) * 100,  # in bp
            'delta_pct': (result1['payer_delta_pct'] + result1['receiver_delta_pct']) * 100,
            'dv01': -result1['dv01'],  # Negative because SOLD
            'interpretation': 'Short a 2Y swap'
        },
        {
            'name': '2 (30Y Payer)',
            'type': 'BUY',
            'strike': swap_rates['30'],
            'forward': swap_rates['30'],
            'moneyness': 0,  # ATM
            'delta_pct': result2['delta_pct'] * 100,
            'dv01': result2['dv01'],
            'interpretation': 'Long 50% of 30Y swap'
        }
    ]
    
    # Print the table
    print_position_summary(positions)


if __name__ == "__main__":
    main()


POSITION SUMMARY
Position             Type     Strike     Forward    Moneyness    Delta %    DV01            Interpretation           
------------------------------------------------------------------------------------------------------------------------
1 (2Y Straddle)      SELL        2.008%    4.043%       -204bp    100.0% $      -28,543 Short a 2Y swap          
2 (30Y Payer)        BUY         4.186%    4.186%          0bp     50.0% $       21,249 Long 50% of 30Y swap     
------------------------------------------------------------------------------------------------------------------------

Net Portfolio Delta:                                         $       -7,294
Interpretation:                                              Short rates (lose if rates rise)


# Find which Tickers Exist and which don't

In [30]:
from xbbg import blp

def find_us_swap_tickers():
    """
    Try different US swap ticker formats to find what works
    """
    
    print("=" * 80)
    print("SEARCHING FOR US SWAP RATE TICKERS")
    print("=" * 80)
    
    # Different possible formats for 2Y US swap
    ticker_formats = [
        # Format 1: Standard swap tickers
        'USSW2 Curncy',
        'USSWAP2 Curncy',
        'USSP2 Curncy',
        
        # Format 2: Yield curve tickers
        'YCSW0002 Index',
        'USYC2Y Index',
        
        # Format 3: Generic tickers
        'USD2Y Curncy',
        'US0002Y Index',
        
        # Format 4: ICAP/BGN tickers
        'USDR2 Curncy',
        'USDR2T Curncy',
        
        # Format 5: CME tickers
        'SRUS2 Comdty',
        
        # Format 6: OIS (if you need OIS instead)
        'USOSFR2 Curncy',
        'USSOFR2 Curncy',
    ]
    
    working_tickers = []
    
    for ticker in ticker_formats:
        print(f"\nTrying: {ticker}")
        try:
            # Check if ticker exists
            name = blp.bdp(ticker, 'NAME')
            
            if not name.empty and name.iloc[0, 0] is not None:
                print(f"  ✓ Ticker exists: {name.iloc[0, 0]}")
                
                # Try to get price
                price = blp.bdp(ticker, 'PX_LAST')
                if not price.empty and price.iloc[0, 0] is not None:
                    print(f"  ✓ Price: {price.iloc[0, 0]}")
                    working_tickers.append(ticker)
                else:
                    print(f"  ⚠ Ticker exists but no price data")
            else:
                print(f"  ✗ Ticker doesn't exist")
                
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print("\n" + "=" * 80)
    if working_tickers:
        print(f"✓ FOUND {len(working_tickers)} WORKING TICKERS:")
        for ticker in working_tickers:
            print(f"  - {ticker}")
    else:
        print("✗ NO WORKING SWAP TICKERS FOUND")
        print("\nPossible reasons:")
        print("1. You don't have swap data subscription in Bloomberg")
        print("2. Need to use different data source (e.g., ICAP, BGN)")
        print("3. Need to access via SWPM or IRSB instead")
    
    return working_tickers


def test_all_tenors(base_format):
    """
    Once we find the right format, test all tenors
    """
    
    print("\n" + "=" * 80)
    print(f"TESTING ALL TENORS WITH FORMAT: {base_format}")
    print("=" * 80)
    
    tenors = ['1', '2', '3', '5', '7', '10', '15', '20', '30']
    
    # Replace '2' with placeholder
    format_template = base_format.replace('2', '{}')
    
    working_tenors = {}
    
    for tenor in tenors:
        ticker = format_template.format(tenor)
        print(f"\nTrying {tenor}Y: {ticker}")
        
        try:
            price = blp.bdp(ticker, 'PX_LAST')
            if not price.empty and price.iloc[0, 0] is not None:
                rate = price.iloc[0, 0]
                print(f"  ✓ {tenor}Y rate: {rate:.3f}%")
                working_tenors[tenor] = {
                    'ticker': ticker,
                    'rate': rate
                }
        except Exception as e:
            print(f"  ✗ Failed")
    
    return working_tenors


def get_swap_curve_alternative():
    """
    Alternative: Get swap curve from SWPM
    """
    
    print("\n" + "=" * 80)
    print("ALTERNATIVE: USING SWPM CURVE")
    print("=" * 80)
    
    print("""
If individual swap tickers don't work, you can:

1. In Bloomberg Terminal, type: SWPM<GO>
2. Select "Swap"
3. Currency: USD
4. Build a standard swap
5. Click "Curves" button
6. Export the curve to Excel
7. Or note the curve name (e.g., "USD SOFR Curve")

Then use that curve name in Python:
    """)
    
    # Try to get curve data
    curve_names = [
        'S0045FS Index',  # USD SOFR Swap Curve
        'YCSW0023 Index', # USD Swap Curve
    ]
    
    for curve in curve_names:
        print(f"\nTrying curve: {curve}")
        try:
            data = blp.bdp(curve, ['PX_LAST', 'NAME'])
            if not data.empty:
                print(f"  ✓ Found: {data}")
        except Exception as e:
            print(f"  ✗ Error: {e}")


if __name__ == "__main__":
    # Step 1: Find working swap tickers
    working = find_us_swap_tickers()
    
    # Step 2: If we found any, test all tenors
    if working:
        print("\n" + "=" * 80)
        print("Testing all tenors with working format...")
        base_ticker = working[0]
        tenors = test_all_tenors(base_ticker)
        
        if tenors:
            print("\n" + "=" * 80)
            print("SUCCESS! Here's your swap curve:")
            print("=" * 80)
            for tenor, data in tenors.items():
                print(f"{tenor}Y: {data['rate']:.3f}% ({data['ticker']})")
    else:
        # Step 3: Try alternative methods
        get_swap_curve_alternative()
    
    print("\n" + "=" * 80)
    print("MANUAL STEPS TO TRY IN BLOOMBERG:")
    print("=" * 80)
    print("""
1. Type: IRSB<GO>
   - Build a 2Y USD swap
   - Look at the ticker shown
   - Try that ticker in Python

2. Type: SWPM<GO>
   - Build a swap
   - Right-click on the swap rate
   - Select "Ticker" to see the Bloomberg ticker

3. Type: ALLX<GO>
   - Search: "USD 2 year swap rate"
   - See what tickers are suggested

4. Check your Bloomberg subscription:
   - Type: BBLS<GO>
   - Check if you have "Interest Rate Derivatives" data

5. Ask your Bloomberg rep:
   - What's the correct ticker for USD swap rates?
   - Do you have ICAP, BGN, or other swap data?
    """)

SEARCHING FOR US SWAP RATE TICKERS

Trying: USSW2 Curncy
  ✓ Ticker exists: USD SWAP SEMI 30/360 2Y
  ⚠ Ticker exists but no price data

Trying: USSWAP2 Curncy
  ✓ Ticker exists: USD SWAP SA 30/360 2YR*
  ⚠ Ticker exists but no price data

Trying: USSP2 Curncy
  ✓ Ticker exists: USD SWAP SPREAD SEMI 2YR
  ✓ Price: 33.1509

Trying: YCSW0002 Index
  ✗ Ticker doesn't exist

Trying: USYC2Y Index
  ✗ Ticker doesn't exist

Trying: USD2Y Curncy
  ✗ Ticker doesn't exist

Trying: US0002Y Index
  ✗ Ticker doesn't exist

Trying: USDR2 Curncy
  ✓ Ticker exists: USD DEPOSIT  2 YR
  ✓ Price: 4.52

Trying: USDR2T Curncy
  ✓ Ticker exists: USD DEPOSIT T/N
  ✓ Price: 3.6625

Trying: SRUS2 Comdty
  ✗ Ticker doesn't exist

Trying: USOSFR2 Curncy
  ✓ Ticker exists: USD OIS  ANN VS SOFR 2Y
  ✓ Price: 4.0231

Trying: USSOFR2 Curncy
  ✗ Ticker doesn't exist

✓ FOUND 4 WORKING TICKERS:
  - USSP2 Curncy
  - USDR2 Curncy
  - USDR2T Curncy
  - USOSFR2 Curncy

Testing all tenors with working format...

TESTING AL